<a href="https://colab.research.google.com/github/BrunoDrezza/double-deep-Q-Learning/blob/main/main_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Notebook para Treino da Rede Neural

> Utilizado para conseguir treinar o modelo no colab, onde tem mais poder computacional

In [ ]:
## Imports
import os
import sys
import tensorflow as tf
from IPython.display import Image, display
# Importa as minhas classes
from src.agents.Agent_NN import DoubleDQNAgent_LunarLander
from src.plots.Plots import plot_learning_curves

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Montagem do Google Drive e Instalação das Dependências

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = "/content/drive/MyDrive/RL_LunarLander"

os.makedirs(f"{DRIVE_DIR}/data", exist_ok=True)
os.makedirs(f"{DRIVE_DIR}/plots/results", exist_ok=True)
print(f"Pasta no Drive pronta: {DRIVE_DIR}")

In [ ]:
!pip install gymnasium[box2d] imageio tensorflow pandas matplotlib seaborn

### Orquestrador ("Main")

In [ ]:
# Garante que o Python ache a pasta src que você arrastou
sys.path.append(os.path.abspath('./'))

# Caminhos no Google Drive
DRIVE_DIR = "/content/drive/MyDrive/RL_LunarLander"
DATA_DIR = f"{DRIVE_DIR}/data"
PLOTS_DIR = f"{DRIVE_DIR}/plots/results"

# Cria as pastas de destino
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)

# ATENÇÃO: Ajuste os episódios se quiser fazer um teste rápido antes (ex: 50)
EPISODES = 1000

scenarios = {
    "DQN Baseline (C=0)": 0,
    "DDQN (C=1)": 1,
    "DDQN (C=10)": 10,
    "DDQN (C=50)": 50
}

histories = {}

for label, c_val in scenarios.items():
    print(f"\n{'='*60}\nEXECUTANDO RUN: {label}\n{'='*60}")

    # Gera os GIFs apenas na run do C=10
    save_gifs = (c_val == 10)

    agent = DoubleDQNAgent_LunarLander(env_name='LunarLander-v3')

    # Inicia o treino (Você verá os prints de cada episódio aparecendo aqui)
    rewards = agent.train(episodes=EPISODES, target_update_freq=c_val, max_steps=1500, save_gifs=save_gifs)
    histories[label] = rewards

    # Salva os pesos da rede no Drive
    filename = label.replace(" ", "_").replace("(", "").replace(")", "").replace("=", "")
    agent.save_weights(f"{DATA_DIR}/{filename}.h5")

    # Limpa a memória da GPU para a próxima run não misturar
    tf.keras.backend.clear_session()

print("\n" + "="*50)
print("TREINAMENTO CONCLUÍDO! GERANDO RESULTADOS...")
print("="*50)

### Resultados

In [ ]:
# Gera e plota o gráfico
plot_path = f"{PLOTS_DIR}/lunarlander_comparison.png"
plot_learning_curves(histories, plot_path)

# Exibe na tela
print("--- GRÁFICO DE CONVERGÊNCIA ---")
display(Image(plot_path))

print("\n--- COMPORTAMENTO APÓS TREINO (GIF) ---")
try:
    display(Image("agente_bom.gif"))
except:
    print("GIF ainda não gerado.")

### Validação de Modelo Salvo
Carrega os pesos `.h5` do Drive e gera um GIF para confirmar que a política foi preservada.

In [ ]:
# Carrega pesos do Drive e gera GIF de validação
weights_path = f"{DATA_DIR}/DDQN_C10.h5"

validation_agent = DoubleDQNAgent_LunarLander(env_name='LunarLander-v3')
validation_agent.load_weights(weights_path)
print(f"Pesos carregados de: {weights_path}")

gif_path = f"{PLOTS_DIR}/validacao_modelo.gif"
validation_agent.generate_gif(gif_path)
print(f"GIF de validação salvo em: {gif_path}")

display(Image(gif_path))

tf.keras.backend.clear_session()
del validation_agent